# Chapter 11.3: Debugging Techniques

This notebook covers debugging techniques for vLLM and SGLang development,
from Python-level debugging to CUDA kernel debugging.

**Learning Objectives:**
- Use pdb/ipdb for Python debugging
- Use GDB and CUDA-GDB for native code debugging
- Profile Python code with py-spy
- Use torch.profiler for PyTorch operations
- Debug common CUDA errors
- Debug NCCL issues in distributed settings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part11/chapter_11.3_debugging.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part11/chapter_11.3_debugging.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Python Debugging with pdb/ipdb

In [ ]:
# Demo: Using pdb/ipdb with vLLM components

# Method 1: Inserting breakpoints in code
pdb_examples = '''
# Method 1: Python built-in breakpoint (Python 3.7+)
def schedule_step(self):
    """Example: debugging the scheduler."""
    breakpoint()  # Drops into pdb here
    # Or equivalently:
    # import pdb; pdb.set_trace()
    
    # In the debugger, you can:
    # n          - next line
    # s          - step into function
    # c          - continue
    # p variable - print variable
    # pp expr    - pretty-print expression
    # l          - list source code
    # w          - show call stack
    # u/d        - go up/down the stack
    # b file:line - set breakpoint
    # h          - help

# Method 2: Using ipdb for better experience
def model_forward(self, input_ids):
    import ipdb; ipdb.set_trace()
    # ipdb provides:
    # - Tab completion
    # - Syntax highlighting
    # - Better stack traces

# Method 3: Conditional breakpoint
def process_request(self, request):
    if request.num_tokens > 512:  # Only break for long sequences
        breakpoint()
    # ... rest of processing

# Method 4: Post-mortem debugging
# Run with: python -m pdb -c continue your_script.py
# This drops into pdb automatically when an exception occurs.

# Method 5: Remote debugging with debugpy
import debugpy
debugpy.listen(("0.0.0.0", 5678))
print("Waiting for debugger to attach...")
debugpy.wait_for_client()
# Now connect VSCode to localhost:5678
'''

print("PDB/IPDB Debugging Techniques for vLLM")
print("=" * 60)
print(pdb_examples)

In [ ]:
# Demo: Practical debugging session - simulating a scheduling bug

from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class DebugRequest:
    request_id: str
    prompt_len: int
    max_tokens: int
    generated_tokens: int = 0
    
    @property
    def total_tokens(self):
        return self.prompt_len + self.generated_tokens
    
    @property
    def is_complete(self):
        return self.generated_tokens >= self.max_tokens

class BuggyScheduler:
    """A scheduler with an intentional bug for debugging practice."""
    
    def __init__(self, max_tokens_per_batch=100):
        self.max_tokens_per_batch = max_tokens_per_batch
        self.waiting = []
        self.running = []
    
    def add_request(self, request):
        self.waiting.append(request)
    
    def step(self):
        """Run one scheduling step. Contains a bug!"""
        # Remove completed requests
        completed = [r for r in self.running if r.is_complete]
        self.running = [r for r in self.running if not r.is_complete]
        
        # Generate one token for running requests
        for r in self.running:
            r.generated_tokens += 1
        
        # Admit new requests
        total_tokens = sum(r.total_tokens for r in self.running)
        
        new_running = []
        still_waiting = []
        for r in self.waiting:
            # BUG: should check total_tokens + r.total_tokens, not r.prompt_len
            if total_tokens + r.prompt_len <= self.max_tokens_per_batch:
                new_running.append(r)
                total_tokens += r.prompt_len  # BUG: should be r.total_tokens
            else:
                still_waiting.append(r)
        
        self.running.extend(new_running)
        self.waiting = still_waiting
        
        return {
            "completed": [r.request_id for r in completed],
            "scheduled": [r.request_id for r in new_running],
            "running": len(self.running),
            "waiting": len(self.waiting),
            "total_tokens": sum(r.total_tokens for r in self.running),
        }

# Run the buggy scheduler and observe the issue
scheduler = BuggyScheduler(max_tokens_per_batch=50)

# Add requests
scheduler.add_request(DebugRequest("r1", prompt_len=20, max_tokens=5))
scheduler.add_request(DebugRequest("r2", prompt_len=15, max_tokens=5))
scheduler.add_request(DebugRequest("r3", prompt_len=25, max_tokens=5))

print("Debugging Session: BuggyScheduler")
print("=" * 60)
print("Constraint: max_tokens_per_batch = 50")
print()

for step in range(8):
    result = scheduler.step()
    actual_total = sum(r.total_tokens for r in scheduler.running)
    violation = actual_total > scheduler.max_tokens_per_batch
    
    print(f"Step {step}: scheduled={result['scheduled']}, "
          f"completed={result['completed']}, "
          f"running={result['running']}, "
          f"total_tokens={actual_total}"
          f"{' *** VIOLATION!' if violation else ''}")

print("\nNotice: The scheduler may exceed the token budget!")
print("Can you find the bug? (Hint: check how total_tokens is computed in step())")

## 2. GDB for Debugging CUDA Kernels

In [ ]:
# Demo: GDB and CUDA-GDB usage for debugging native code

gdb_guide = '''
=======================================================
GDB for Debugging C++/CUDA Extensions in vLLM
=======================================================

1. Build with debug symbols:
   export CMAKE_BUILD_TYPE=Debug
   pip install -e . --no-build-isolation

2. Run vLLM under GDB:
   gdb -ex run --args python -m vllm.entrypoints.openai.api_server \\
       --model facebook/opt-125m --enforce-eager

3. Common GDB commands for vLLM debugging:
   # Set breakpoint in C++ code
   (gdb) break cache_ops.cu:paged_attention_v1_kernel
   
   # Set breakpoint on exception
   (gdb) catch throw
   
   # Print CUDA error
   (gdb) call (char*)cudaGetErrorString(cudaGetLastError())
   
   # Backtrace
   (gdb) bt
   (gdb) bt full  # with local variables
   
   # Print variables
   (gdb) p tensor.sizes()
   (gdb) p tensor.data_ptr<float>()

=======================================================
CUDA-GDB for GPU Kernel Debugging
=======================================================

1. Build with CUDA debug symbols:
   export TORCH_CUDA_ARCH_LIST="8.0"  # Your GPU arch
   export CUDA_NVCC_FLAGS="-g -G"     # Debug + device debug
   pip install -e . --no-build-isolation

2. Run under cuda-gdb:
   cuda-gdb --args python your_script.py

3. CUDA-GDB specific commands:
   # Set breakpoint in CUDA kernel
   (cuda-gdb) break paged_attention_kernel
   
   # Switch between GPU threads
   (cuda-gdb) cuda thread (0,0,0)      # Switch to thread (0,0,0)
   (cuda-gdb) cuda block (1,0,0)       # Switch to block (1,0,0)
   
   # Print GPU thread info
   (cuda-gdb) info cuda threads
   (cuda-gdb) info cuda blocks
   (cuda-gdb) info cuda kernels
   
   # Print shared memory
   (cuda-gdb) print shared_mem[threadIdx.x]
   
   # Check for memory errors
   (cuda-gdb) set cuda memcheck on

4. Environment variables for CUDA debugging:
   export CUDA_LAUNCH_BLOCKING=1       # Synchronous kernel launches
   export TORCH_USE_CUDA_DSA=1         # Device-side assertions
   export CUDA_VISIBLE_DEVICES=0       # Single GPU for debugging
'''

print(gdb_guide)

In [ ]:
# Demo: Setting up debug builds and environment

debug_build_script = '''
#!/bin/bash
# debug_build.sh - Build vLLM with full debug symbols

set -e

# 1. Set debug build type
export CMAKE_BUILD_TYPE=Debug

# 2. Enable CUDA debug symbols (makes kernels debuggable)
export CUDA_NVCC_FLAGS="-g -G -O0"

# 3. Set GPU architecture (check yours with nvidia-smi)
export TORCH_CUDA_ARCH_LIST="8.0"  # A100
# export TORCH_CUDA_ARCH_LIST="7.0"  # V100
# export TORCH_CUDA_ARCH_LIST="9.0"  # H100

# 4. Reduce parallel jobs (debug builds use more memory)
export MAX_JOBS=2

# 5. Build
pip install -e . --no-build-isolation

echo "Debug build complete!"
echo "Run with: CUDA_LAUNCH_BLOCKING=1 python your_script.py"
'''

print("Debug Build Script:")
print(debug_build_script)

# Environment variables for debugging
debug_env_vars = {
    "CUDA_LAUNCH_BLOCKING": {
        "value": "1",
        "purpose": "Make CUDA operations synchronous for precise error locations",
        "when": "Debugging CUDA errors (illegal access, assertions)"
    },
    "TORCH_USE_CUDA_DSA": {
        "value": "1",
        "purpose": "Enable device-side assertions in PyTorch",
        "when": "Debugging assertion failures in CUDA kernels"
    },
    "NCCL_DEBUG": {
        "value": "INFO",
        "purpose": "Enable NCCL debug logging",
        "when": "Debugging distributed communication issues"
    },
    "VLLM_LOGGING_LEVEL": {
        "value": "DEBUG",
        "purpose": "Enable detailed vLLM logging",
        "when": "Tracking request flow through the engine"
    },
    "VLLM_TRACE_FUNCTION": {
        "value": "1",
        "purpose": "Trace function calls in vLLM",
        "when": "Understanding execution flow"
    },
}

print("\nDebug Environment Variables:")
print("=" * 70)
for var, info in debug_env_vars.items():
    print(f"\n  {var}={info['value']}")
    print(f"    Purpose: {info['purpose']}")
    print(f"    When:    {info['when']}")

## 3. Profiling with py-spy

In [ ]:
# Demo: Using py-spy for Python profiling

pyspy_guide = '''
=======================================================
py-spy: Sampling Profiler for Python
=======================================================

Install:
  pip install py-spy

Usage 1: Record a profile (creates a flamegraph SVG)
  sudo py-spy record -o profile.svg -- python -m vllm.entrypoints.openai.api_server \\
      --model facebook/opt-125m --port 8000

Usage 2: Top-like view of a running process
  # Find the PID of the vLLM process
  pgrep -f vllm
  
  # Attach to it
  sudo py-spy top --pid <PID>

Usage 3: Dump current call stacks
  sudo py-spy dump --pid <PID>

Usage 4: Record with specific duration
  sudo py-spy record -o profile.svg --duration 30 --pid <PID>

Usage 5: Record with subprocesses (for Ray workers)
  sudo py-spy record -o profile.svg --subprocesses -- python script.py

Key flags:
  --native       Include C/C++ frames (useful for CUDA)
  --rate 100     Sampling rate (samples per second)
  --format       Output format: flamegraph, speedscope, raw
  --idle         Include idle threads
  --gil          Only show threads holding the GIL

Example workflow for vLLM:
  1. Start vLLM server
  2. Start py-spy recording
  3. Send a batch of requests
  4. Stop recording
  5. Open profile.svg in browser
  6. Look for hot spots (wide bars in flamegraph)
'''

print(pyspy_guide)

In [ ]:
# Demo: Simulated profiling to identify bottlenecks
import time
import random

class SimulatedProfiler:
    """Simulates profiling results to teach interpretation."""
    
    def __init__(self):
        self.samples = []
    
    def simulate_vllm_profile(self, num_samples=1000):
        """Generate simulated profiling data for a vLLM workload."""
        functions = [
            ("torch.nn.Linear.forward", 0.25),
            ("vllm.attention.paged_attention", 0.20),
            ("torch.cuda.synchronize", 0.15),
            ("vllm.core.scheduler.schedule", 0.10),
            ("vllm.worker.model_runner.execute", 0.08),
            ("tokenizer.encode", 0.07),
            ("vllm.engine.process_request", 0.05),
            ("torch.Tensor.contiguous", 0.04),
            ("vllm.sampling.sample", 0.03),
            ("networking/asyncio", 0.03),
        ]
        
        self.samples = []
        for func, weight in functions:
            count = int(num_samples * weight)
            self.samples.extend([(func, weight)] * count)
        
        random.shuffle(self.samples)
    
    def print_top_functions(self, top_n=10):
        """Print the top functions by sample count (like py-spy top)."""
        from collections import Counter
        counts = Counter(s[0] for s in self.samples)
        total = len(self.samples)
        
        print(f"{'Function':50s} {'Samples':>8s} {'%':>6s}")
        print("-" * 66)
        for func, count in counts.most_common(top_n):
            pct = 100.0 * count / total
            bar = '#' * int(pct / 2)
            print(f"{func:50s} {count:8d} {pct:5.1f}% {bar}")
    
    def categorize_bottlenecks(self):
        """Categorize bottlenecks into compute, memory, and I/O."""
        categories = {
            "GPU Compute": ["torch.nn.Linear.forward", "vllm.attention.paged_attention",
                           "vllm.sampling.sample"],
            "GPU Sync/Wait": ["torch.cuda.synchronize"],
            "Scheduling": ["vllm.core.scheduler.schedule", "vllm.engine.process_request"],
            "Data Movement": ["torch.Tensor.contiguous", "tokenizer.encode"],
            "I/O": ["networking/asyncio"],
            "Execution": ["vllm.worker.model_runner.execute"],
        }
        
        from collections import Counter
        counts = Counter(s[0] for s in self.samples)
        total = len(self.samples)
        
        print("\nBottleneck Categories:")
        print("=" * 50)
        for category, funcs in categories.items():
            cat_count = sum(counts.get(f, 0) for f in funcs)
            pct = 100.0 * cat_count / total
            bar = '#' * int(pct)
            print(f"  {category:20s}: {pct:5.1f}% {bar}")

profiler = SimulatedProfiler()
profiler.simulate_vllm_profile()

print("Simulated py-spy Profile for vLLM Serving Workload")
print("=" * 66)
profiler.print_top_functions()
profiler.categorize_bottlenecks()

## 4. torch.profiler for PyTorch Operations

In [ ]:
# Demo: Using torch.profiler with vLLM/SGLang

torch_profiler_code = '''
import torch
from torch.profiler import profile, record_function, ProfilerActivity

# === Method 1: Basic profiling ===
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
) as prof:
    # Run your vLLM code here
    from vllm import LLM, SamplingParams
    llm = LLM(model="facebook/opt-125m", max_model_len=256)
    output = llm.generate(
        ["Hello world"],
        SamplingParams(max_tokens=10, temperature=0)
    )

# Print summary sorted by CUDA time
print(prof.key_averages().table(
    sort_by="cuda_time_total",
    row_limit=20
))

# Export for Chrome trace viewer
prof.export_chrome_trace("vllm_trace.json")

# Export for TensorBoard
prof.export_stacks("vllm_stacks.txt", "self_cuda_time_total")


# === Method 2: Scheduled profiling (skip warmup) ===
def trace_handler(prof):
    """Called after each profiling step."""
    print(prof.key_averages().table(
        sort_by="self_cuda_time_total", row_limit=10
    ))
    prof.export_chrome_trace(f"trace_{prof.step_num}.json")

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule=torch.profiler.schedule(
        wait=2,    # Skip first 2 steps (warmup)
        warmup=1,  # Warmup for 1 step  
        active=3,  # Profile 3 steps
        repeat=1,  # Run this schedule once
    ),
    on_trace_ready=trace_handler,
    with_stack=True,
) as prof:
    for step in range(6):
        # Run inference step
        output = llm.generate(
            ["Test prompt"],
            SamplingParams(max_tokens=10)
        )
        prof.step()  # Signal end of step


# === Method 3: Custom annotations ===
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    with record_function("tokenization"):
        tokens = tokenizer.encode("Hello world")
    
    with record_function("model_forward"):
        output = model(tokens)
    
    with record_function("sampling"):
        next_token = sample(output.logits)
'''

print("torch.profiler Usage Guide for vLLM/SGLang")
print("=" * 60)
print(torch_profiler_code)

In [ ]:
# Demo: Simulated torch.profiler output analysis

class SimulatedTorchProfile:
    """Simulates torch.profiler output for learning."""
    
    def __init__(self):
        self.operations = [
            {"name": "aten::linear", "cpu_time_us": 150, "cuda_time_us": 2500,
             "calls": 96, "input_shapes": "[[1,128,768], [768,768]]"},
            {"name": "aten::scaled_dot_product_attention", "cpu_time_us": 80,
             "cuda_time_us": 1800, "calls": 12, "input_shapes": "[[1,12,128,64]]"},
            {"name": "aten::layer_norm", "cpu_time_us": 50, "cuda_time_us": 400,
             "calls": 24, "input_shapes": "[[1,128,768]]"},
            {"name": "aten::gelu", "cpu_time_us": 20, "cuda_time_us": 300,
             "calls": 12, "input_shapes": "[[1,128,3072]]"},
            {"name": "aten::embedding", "cpu_time_us": 30, "cuda_time_us": 100,
             "calls": 1, "input_shapes": "[[50272,768], [1,128]]"},
            {"name": "aten::softmax", "cpu_time_us": 15, "cuda_time_us": 200,
             "calls": 1, "input_shapes": "[[1,50272]]"},
            {"name": "aten::copy_", "cpu_time_us": 100, "cuda_time_us": 50,
             "calls": 200, "input_shapes": "various"},
            {"name": "cudaLaunchKernel", "cpu_time_us": 500, "cuda_time_us": 0,
             "calls": 350, "input_shapes": "N/A"},
            {"name": "cudaDeviceSynchronize", "cpu_time_us": 3000, "cuda_time_us": 0,
             "calls": 5, "input_shapes": "N/A"},
        ]
    
    def print_table(self, sort_by="cuda_time_us"):
        sorted_ops = sorted(self.operations, key=lambda x: x[sort_by], reverse=True)
        
        print(f"{'Operation':45s} {'CPU(us)':>8s} {'CUDA(us)':>9s} {'Calls':>6s} {'Shapes'}")
        print("-" * 100)
        for op in sorted_ops:
            total_cpu = op['cpu_time_us'] * op['calls']
            total_cuda = op['cuda_time_us'] * op['calls']
            print(f"{op['name']:45s} {total_cpu:8d} {total_cuda:9d} {op['calls']:6d} {op['input_shapes']}")
    
    def analyze(self):
        total_cuda = sum(op['cuda_time_us'] * op['calls'] for op in self.operations)
        total_cpu = sum(op['cpu_time_us'] * op['calls'] for op in self.operations)
        
        print("\nAnalysis:")
        print(f"  Total CPU time:  {total_cpu:,} us")
        print(f"  Total CUDA time: {total_cuda:,} us")
        print(f"  GPU utilization:  ~{100*total_cuda/(total_cuda+total_cpu):.0f}%")
        
        # Top bottleneck
        top = max(self.operations, key=lambda x: x['cuda_time_us'] * x['calls'])
        pct = 100 * top['cuda_time_us'] * top['calls'] / total_cuda
        print(f"\n  Top CUDA bottleneck: {top['name']} ({pct:.1f}% of CUDA time)")
        
        # Kernel launch overhead
        launch_ops = [op for op in self.operations if 'Launch' in op['name']]
        if launch_ops:
            launch_time = sum(op['cpu_time_us'] * op['calls'] for op in launch_ops)
            print(f"  Kernel launch overhead: {launch_time:,} us ({100*launch_time/total_cpu:.1f}% of CPU)")

profile = SimulatedTorchProfile()
print("Simulated torch.profiler Output")
print("=" * 100)
profile.print_table(sort_by="cuda_time_us")
profile.analyze()

## 5. CUDA Error Debugging

In [ ]:
# Demo: Common CUDA errors and how to debug them

cuda_errors = [
    {
        "error": "CUDA error: an illegal memory access was encountered",
        "common_causes": [
            "Out-of-bounds array access in CUDA kernel",
            "Accessing freed GPU memory",
            "Mismatched tensor shapes passed to kernel",
            "Race condition in kernel"
        ],
        "debug_steps": [
            "1. Set CUDA_LAUNCH_BLOCKING=1 to get exact error location",
            "2. Run with compute-sanitizer: compute-sanitizer python script.py",
            "3. Check tensor shapes before kernel launch",
            "4. Use cuda-gdb with memcheck: set cuda memcheck on"
        ]
    },
    {
        "error": "CUDA out of memory (OOM)",
        "common_causes": [
            "Model too large for GPU memory",
            "KV cache too large",
            "Memory fragmentation",
            "Memory leak in custom kernels"
        ],
        "debug_steps": [
            "1. Check memory with torch.cuda.memory_summary()",
            "2. Reduce max_model_len or gpu_memory_utilization",
            "3. Enable memory profiling: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True",
            "4. Use torch.cuda.memory_snapshot() for detailed analysis"
        ]
    },
    {
        "error": "RuntimeError: CUDA error: device-side assert triggered",
        "common_causes": [
            "Index out of range (e.g., token ID >= vocab_size)",
            "NaN in computation",
            "Invalid configuration parameter"
        ],
        "debug_steps": [
            "1. Set CUDA_LAUNCH_BLOCKING=1",
            "2. Set TORCH_USE_CUDA_DSA=1 for device-side assertion messages",
            "3. Check input token IDs are within vocab range",
            "4. Check for NaN values: torch.isnan(tensor).any()"
        ]
    },
    {
        "error": "NCCL error: unhandled system error / connection refused",
        "common_causes": [
            "Network misconfiguration in distributed setup",
            "Firewall blocking NCCL ports",
            "Incorrect NCCL_SOCKET_IFNAME",
            "Version mismatch between nodes"
        ],
        "debug_steps": [
            "1. Set NCCL_DEBUG=INFO for detailed logging",
            "2. Set NCCL_DEBUG_SUBSYS=ALL for all subsystems",
            "3. Test basic NCCL: python -m torch.distributed.run --nproc_per_node=2 test.py",
            "4. Check NCCL_SOCKET_IFNAME matches your network interface"
        ]
    }
]

print("Common CUDA Errors and Debugging Guide")
print("=" * 70)
for i, err in enumerate(cuda_errors, 1):
    print(f"\n--- Error {i} ---")
    print(f"Error: {err['error']}")
    print(f"\nCommon Causes:")
    for cause in err['common_causes']:
        print(f"  - {cause}")
    print(f"\nDebug Steps:")
    for step in err['debug_steps']:
        print(f"  {step}")

In [ ]:
# Demo: Memory debugging utilities

memory_debug_code = '''
import torch

def debug_gpu_memory(label=""):
    """Print detailed GPU memory information."""
    if not torch.cuda.is_available():
        print("CUDA not available")
        return
    
    for i in range(torch.cuda.device_count()):
        print(f"\n=== GPU {i}: {torch.cuda.get_device_name(i)} ===")
        if label:
            print(f"Label: {label}")
        
        # Memory allocated by PyTorch
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        reserved = torch.cuda.memory_reserved(i) / 1024**3
        max_allocated = torch.cuda.max_memory_allocated(i) / 1024**3
        
        # Total GPU memory
        total = torch.cuda.get_device_properties(i).total_mem / 1024**3
        
        print(f"  Total:         {total:.2f} GB")
        print(f"  Allocated:     {allocated:.2f} GB ({100*allocated/total:.1f}%)")
        print(f"  Reserved:      {reserved:.2f} GB ({100*reserved/total:.1f}%)")
        print(f"  Max allocated: {max_allocated:.2f} GB")
        print(f"  Free:          {total-reserved:.2f} GB")


def find_memory_leaks():
    """Track memory allocations to find leaks."""
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    
    baseline = torch.cuda.memory_allocated()
    print(f"Baseline: {baseline / 1024**2:.1f} MB")
    
    # Run your code here
    # ...
    
    after = torch.cuda.memory_allocated()
    leaked = after - baseline
    print(f"After:    {after / 1024**2:.1f} MB")
    print(f"Leaked:   {leaked / 1024**2:.1f} MB")
    
    if leaked > 0:
        print("\nMemory snapshot:")
        snapshot = torch.cuda.memory_snapshot()
        for block in snapshot[:5]:  # Show first 5 blocks
            print(f"  Size: {block.get('size', 0)/1024:.1f} KB, "
                  f"State: {block.get('state', 'unknown')}")


# Usage in vLLM debugging:
# 1. Add debug_gpu_memory("before_forward") before model forward
# 2. Add debug_gpu_memory("after_forward") after model forward  
# 3. Compare to see where memory is used
'''

print("GPU Memory Debugging Utilities")
print("=" * 60)
print(memory_debug_code)

## 6. NCCL Debugging for Distributed Issues

In [ ]:
# Demo: NCCL debugging guide for distributed vLLM

nccl_guide = '''
=======================================================
NCCL Debugging Guide for Distributed vLLM/SGLang
=======================================================

Environment Variables:
  export NCCL_DEBUG=INFO         # Basic debug info
  export NCCL_DEBUG=TRACE        # Detailed traces
  export NCCL_DEBUG_SUBSYS=ALL   # All subsystems
  export NCCL_DEBUG_FILE=/tmp/nccl_%h_%p.log  # Per-process log

  # Network configuration
  export NCCL_SOCKET_IFNAME=eth0  # Network interface
  export NCCL_IB_DISABLE=1        # Disable InfiniBand
  export NCCL_P2P_DISABLE=1       # Disable P2P (for debugging)
  export NCCL_SHM_DISABLE=1       # Disable shared memory

Common NCCL Issues:

1. Timeout / Hang
   - Symptom: Process hangs during all_reduce/all_gather
   - Debug: Set NCCL_DEBUG=TRACE, check if all ranks enter collective
   - Fix: Ensure all ranks call same collective in same order

2. Connection Refused
   - Symptom: NCCL error during init
   - Debug: Check firewall rules, NCCL_SOCKET_IFNAME
   - Fix: Set correct network interface, open required ports

3. Data Corruption
   - Symptom: Different results on different GPUs
   - Debug: Verify inputs to collectives are same shape on all ranks
   - Fix: Add shape assertions before collectives

4. OOM During Communication
   - Symptom: NCCL OOM during all_reduce
   - Debug: Check NCCL buffer sizes
   - Fix: export NCCL_BUFFSIZE=8388608 (reduce buffer size)

Test Script:
  python -m torch.distributed.run \\
    --nproc_per_node=2 \\
    -m vllm.entrypoints.openai.api_server \\
    --model facebook/opt-1.3b \\
    --tensor-parallel-size 2
'''

print(nccl_guide)

In [ ]:
# Demo: Distributed debugging helper class

class DistributedDebugger:
    """Helper for debugging distributed vLLM issues."""
    
    def __init__(self):
        self.rank = 0
        self.world_size = 1
    
    def check_environment(self):
        """Check distributed environment variables."""
        import os
        
        dist_vars = [
            "MASTER_ADDR", "MASTER_PORT",
            "RANK", "WORLD_SIZE", "LOCAL_RANK",
            "NCCL_DEBUG", "NCCL_SOCKET_IFNAME",
            "CUDA_VISIBLE_DEVICES",
        ]
        
        print("Distributed Environment:")
        print("-" * 40)
        for var in dist_vars:
            value = os.environ.get(var, "not set")
            print(f"  {var}: {value}")
    
    def verify_all_reduce(self, tensor_size=1024):
        """Verify NCCL all_reduce works correctly."""
        test_code = f'''
import torch
import torch.distributed as dist

dist.init_process_group(backend="nccl")
rank = dist.get_rank()
device = torch.device(f"cuda:{{rank}}")

# Each rank creates a tensor with its rank value
tensor = torch.full(({tensor_size},), float(rank), device=device)
print(f"Rank {{rank}}: before all_reduce: {{tensor[:5]}}")

dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
print(f"Rank {{rank}}: after all_reduce: {{tensor[:5]}}")

# Expected: sum of all ranks
expected = sum(range(dist.get_world_size()))
assert torch.allclose(tensor, torch.full_like(tensor, expected))
print(f"Rank {{rank}}: PASSED")
dist.destroy_process_group()
'''
        print("NCCL All-Reduce Verification Script:")
        print(test_code)
        print("Run with: torchrun --nproc_per_node=2 test_allreduce.py")
    
    def generate_debug_report(self):
        """Generate a debug report for filing issues."""
        import platform
        import sys
        
        print("\nDebug Report")
        print("=" * 50)
        print(f"Python: {sys.version}")
        print(f"OS: {platform.platform()}")
        
        try:
            import torch
            print(f"PyTorch: {torch.__version__}")
            print(f"CUDA: {torch.version.cuda}")
            print(f"cuDNN: {torch.backends.cudnn.version()}")
            print(f"GPUs: {torch.cuda.device_count()}")
        except ImportError:
            print("PyTorch: not installed")
        
        try:
            import vllm
            print(f"vLLM: {vllm.__version__}")
        except (ImportError, AttributeError):
            print("vLLM: not installed")

debugger = DistributedDebugger()
debugger.check_environment()
print()
debugger.verify_all_reduce()
print()
debugger.generate_debug_report()

## 7. Common Bugs and Diagnostic Patterns

In [ ]:
# Demo: Common bug patterns in LLM serving systems

class BugPatternAnalyzer:
    """Analyze common bug patterns in vLLM/SGLang."""
    
    def __init__(self):
        self.patterns = [
            {
                "name": "Token ID Out of Range",
                "symptoms": ["CUDA device-side assert", "IndexError in embedding"],
                "check": self._check_token_range,
                "fix": "Verify tokenizer vocab_size matches model config"
            },
            {
                "name": "KV Cache Shape Mismatch",
                "symptoms": ["Illegal memory access", "Wrong output shapes"],
                "check": self._check_kv_cache_shapes,
                "fix": "Check num_heads, head_dim, num_layers match model"
            },
            {
                "name": "Sequence Length Overflow",
                "symptoms": ["OOM", "Garbage output for long sequences"],
                "check": self._check_seq_length,
                "fix": "Set max_model_len appropriately"
            },
            {
                "name": "dtype Mismatch",
                "symptoms": ["NaN in output", "Precision loss", "Wrong results"],
                "check": self._check_dtype,
                "fix": "Ensure consistent dtype across model, cache, and inputs"
            },
        ]
    
    def _check_token_range(self, token_ids, vocab_size):
        invalid = [t for t in token_ids if t < 0 or t >= vocab_size]
        return len(invalid) == 0, f"Invalid tokens: {invalid[:5]}"
    
    def _check_kv_cache_shapes(self, cache_shape, expected_shape):
        match = cache_shape == expected_shape
        return match, f"Got {cache_shape}, expected {expected_shape}"
    
    def _check_seq_length(self, seq_len, max_len):
        valid = seq_len <= max_len
        return valid, f"seq_len={seq_len}, max={max_len}"
    
    def _check_dtype(self, tensor_dtype, expected_dtype):
        match = tensor_dtype == expected_dtype
        return match, f"Got {tensor_dtype}, expected {expected_dtype}"
    
    def diagnose(self, error_message):
        """Suggest debugging steps based on error message."""
        error_lower = error_message.lower()
        matches = []
        for pattern in self.patterns:
            for symptom in pattern["symptoms"]:
                if symptom.lower() in error_lower:
                    matches.append(pattern)
                    break
        return matches

# Test the analyzer
analyzer = BugPatternAnalyzer()

print("Bug Pattern Analyzer")
print("=" * 60)

# Simulate diagnoses
test_errors = [
    "RuntimeError: CUDA error: device-side assert triggered",
    "torch.cuda.OutOfMemoryError: CUDA out of memory",
    "RuntimeError: CUDA error: an illegal memory access was encountered",
    "Output contains NaN values",
]

for error in test_errors:
    print(f"\nError: {error}")
    matches = analyzer.diagnose(error)
    if matches:
        for m in matches:
            print(f"  Possible cause: {m['name']}")
            print(f"  Fix: {m['fix']}")
    else:
        print("  No matching patterns found. Check logs for more details.")

---
## Hands-On Assignments

### Assignment 1: Debug a Simulated Scheduling Bug

**Objective:** Find and fix the bug in the `BuggyTokenBudgetScheduler` below.

The scheduler is supposed to maintain a strict token budget, but it sometimes allows
more tokens than the budget allows.

In [ ]:
# Assignment 1: Find and fix the bug

from dataclasses import dataclass
from typing import List, Dict

@dataclass
class TokenRequest:
    request_id: str
    prompt_tokens: int
    max_new_tokens: int
    generated: int = 0
    
    @property
    def current_tokens(self):
        return self.prompt_tokens + self.generated
    
    @property
    def is_done(self):
        return self.generated >= self.max_new_tokens

class BuggyTokenBudgetScheduler:
    """Scheduler with a bug in token budget tracking.
    
    Bug: The scheduler does not correctly account for tokens that will be
    generated in the current step when admitting new requests.
    
    Your task: 
    1. Run the simulation below and observe the budget violations
    2. Use print-based debugging to find where the violation occurs
    3. Fix the bug
    4. Verify the fix
    """
    
    def __init__(self, token_budget: int):
        self.token_budget = token_budget
        self.waiting: List[TokenRequest] = []
        self.running: List[TokenRequest] = []
    
    def add_request(self, req: TokenRequest):
        self.waiting.append(req)
    
    def step(self) -> Dict:
        # 1. Generate token for running requests
        for req in self.running:
            if not req.is_done:
                req.generated += 1
        
        # 2. Remove completed
        done = [r for r in self.running if r.is_done]
        self.running = [r for r in self.running if not r.is_done]
        
        # 3. Calculate current token usage
        # BUG: This uses prompt_tokens instead of current_tokens
        used_tokens = sum(r.prompt_tokens for r in self.running)
        
        # 4. Admit new requests within budget
        newly_scheduled = []
        remaining = []
        for req in self.waiting:
            if used_tokens + req.prompt_tokens <= self.token_budget:
                newly_scheduled.append(req)
                used_tokens += req.prompt_tokens
            else:
                remaining.append(req)
        
        self.running.extend(newly_scheduled)
        self.waiting = remaining
        
        actual_tokens = sum(r.current_tokens for r in self.running)
        return {
            "scheduled": [r.request_id for r in newly_scheduled],
            "completed": [r.request_id for r in done],
            "running": len(self.running),
            "actual_tokens": actual_tokens,
            "budget": self.token_budget,
            "violation": actual_tokens > self.token_budget
        }

# Run simulation to observe the bug
scheduler = BuggyTokenBudgetScheduler(token_budget=100)
scheduler.add_request(TokenRequest("r1", prompt_tokens=30, max_new_tokens=20))
scheduler.add_request(TokenRequest("r2", prompt_tokens=25, max_new_tokens=15))
scheduler.add_request(TokenRequest("r3", prompt_tokens=20, max_new_tokens=10))
scheduler.add_request(TokenRequest("r4", prompt_tokens=15, max_new_tokens=10))

print("Simulation: Token Budget = 100")
print("=" * 70)
violations = 0
for step in range(25):
    result = scheduler.step()
    v = " *** VIOLATION" if result['violation'] else ""
    if result['violation']:
        violations += 1
    print(f"Step {step:2d}: running={result['running']}, "
          f"tokens={result['actual_tokens']}/{result['budget']}, "
          f"new={result['scheduled']}, done={result['completed']}{v}")

print(f"\nTotal budget violations: {violations}")
print("\nTODO: Fix the bug in BuggyTokenBudgetScheduler.step()")
print("Hint: Look at how 'used_tokens' is calculated in step 3.")

### Assignment 2: Use py-spy to Find a Python Bottleneck

**Objective:** Profile the `SlowProcessor` and identify the bottleneck.

In [ ]:
# Assignment 2: Starter Code
import time
import cProfile
import pstats
import io

class SlowProcessor:
    """A request processor with hidden bottlenecks.
    
    Your task:
    1. Profile this processor using cProfile (since py-spy needs a separate process)
    2. Identify the top 3 bottleneck functions
    3. Suggest optimizations
    """
    
    def __init__(self):
        self.cache = {}
    
    def tokenize(self, text):
        """Simulate tokenization."""
        tokens = []
        for word in text.split():
            # Inefficient: computing hash for each word every time
            token_id = hash(word) % 50000
            tokens.append(token_id)
        return tokens
    
    def _slow_attention(self, tokens):
        """Simulate attention computation (intentionally slow)."""
        n = len(tokens)
        scores = []
        for i in range(n):
            row = []
            for j in range(n):
                # O(n^2) computation
                score = sum(tokens[i:j+1]) / (j - i + 1) if j >= i else 0
                row.append(score)
            scores.append(row)
        return scores
    
    def _apply_softmax(self, scores):
        """Apply softmax to attention scores."""
        import math
        result = []
        for row in scores:
            max_score = max(row) if row else 0
            exp_scores = [math.exp(s - max_score) for s in row]
            total = sum(exp_scores)
            result.append([s / total if total > 0 else 0 for s in exp_scores])
        return result
    
    def _generate_output(self, attention, tokens):
        """Generate output tokens."""
        output = []
        for i in range(min(10, len(tokens))):
            # Weighted sum
            weighted = sum(a * t for a, t in zip(attention[i], tokens))
            output.append(int(weighted) % 50000)
        return output
    
    def process(self, text):
        """Process a request end-to-end."""
        tokens = self.tokenize(text)
        attention = self._slow_attention(tokens)
        probs = self._apply_softmax(attention)
        output = self._generate_output(probs, tokens)
        return output

# Profile the processor
processor = SlowProcessor()
test_text = " ".join([f"word{i}" for i in range(50)])  # 50-word input

print("Profiling SlowProcessor...")
print("=" * 60)

# Use cProfile
pr = cProfile.Profile()
pr.enable()

for _ in range(5):
    processor.process(test_text)

pr.disable()

# Print results
s = io.StringIO()
ps = pstats.Stats(pr, stream=s)
ps.sort_stats('cumulative')
ps.print_stats(15)
print(s.getvalue())

print("\nTODO: Identify the top 3 bottlenecks and suggest optimizations.")
print("Hint: Which function takes the most cumulative time?")

### Assignment 3: Diagnose a CUDA Memory Error

**Objective:** Use debugging techniques to diagnose and fix memory-related bugs.

In [ ]:
# Assignment 3: Starter Code

class MemoryBugSimulator:
    """Simulates common CUDA memory bugs for practice.
    
    Your task:
    1. Run each scenario
    2. Identify the root cause
    3. Suggest the fix
    4. Write a diagnostic function that would catch this bug early
    """
    
    def scenario_1_buffer_overflow(self):
        """Simulates buffer overflow in KV cache."""
        cache_size = 100  # blocks
        block_size = 16   # tokens per block
        
        # Allocate cache
        cache = [None] * cache_size
        allocated = 0
        
        # Simulate requests that exceed cache capacity
        requests = [
            {"id": "r1", "tokens": 500},   # 32 blocks
            {"id": "r2", "tokens": 400},   # 25 blocks
            {"id": "r3", "tokens": 600},   # 38 blocks - should fail!
            {"id": "r4", "tokens": 200},   # 13 blocks
        ]
        
        import math
        for req in requests:
            blocks_needed = math.ceil(req["tokens"] / block_size)
            new_total = allocated + blocks_needed
            
            # BUG: No check for cache overflow!
            if new_total > cache_size:
                print(f"  [ERROR] {req['id']}: needs {blocks_needed} blocks, "
                      f"only {cache_size - allocated} free. OVERFLOW!")
                # In CUDA, this would be an illegal memory access
            else:
                allocated = new_total
                print(f"  [OK] {req['id']}: allocated {blocks_needed} blocks "
                      f"({allocated}/{cache_size} used)")
    
    def scenario_2_use_after_free(self):
        """Simulates use-after-free bug."""
        # Simulate block allocation tracking
        active_blocks = {}
        freed_blocks = set()
        
        # Allocate blocks for requests
        active_blocks["r1"] = [0, 1, 2]
        active_blocks["r2"] = [3, 4, 5]
        
        # Free r1's blocks
        freed_blocks.update(active_blocks["r1"])
        del active_blocks["r1"]
        print(f"  Freed blocks for r1: {freed_blocks}")
        
        # Allocate new request using freed blocks
        active_blocks["r3"] = [0, 1]  # Reuse freed blocks
        freed_blocks -= {0, 1}
        print(f"  Allocated blocks for r3: {active_blocks['r3']}")
        
        # BUG: Still holding reference to old block 2 somewhere
        stale_ref = 2  # This block was freed but reference wasn't cleared
        print(f"  [BUG] Stale reference to block {stale_ref} "
              f"(freed={stale_ref in freed_blocks})")
        print(f"  In CUDA, accessing block {stale_ref} would cause "
              f"'illegal memory access'")
    
    def scenario_3_shape_mismatch(self):
        """Simulates tensor shape mismatch."""
        # Simulate KV cache with wrong dimensions
        num_heads = 12
        head_dim = 64
        num_layers = 12
        
        # Cache allocated with these dims
        cache_shape = (num_layers, 2, 100, num_heads, head_dim)  # [layers, kv, blocks, heads, dim]
        
        # Model loaded with different config (wrong head count)
        model_num_heads = 16  # Different from cache!
        
        if num_heads != model_num_heads:
            print(f"  [BUG] Shape mismatch!")
            print(f"    Cache: num_heads={num_heads}")
            print(f"    Model: num_heads={model_num_heads}")
            print(f"    This would cause incorrect memory access patterns")
            print(f"    Fix: Ensure cache config matches model config")

sim = MemoryBugSimulator()

print("Memory Bug Scenarios")
print("=" * 60)

print("\nScenario 1: Buffer Overflow")
print("-" * 40)
sim.scenario_1_buffer_overflow()

print("\nScenario 2: Use After Free")
print("-" * 40)
sim.scenario_2_use_after_free()

print("\nScenario 3: Shape Mismatch")
print("-" * 40)
sim.scenario_3_shape_mismatch()

print("\nTODO: For each scenario, write a diagnostic check function")
print("that would catch the bug before it causes a CUDA error.")

In [ ]:
# Assignment 3 continued: Write diagnostic functions

class MemoryDiagnostics:
    """TODO: Implement diagnostic checks for each scenario."""
    
    @staticmethod
    def check_cache_capacity(allocated_blocks, total_blocks, requested_blocks):
        """TODO: Check if allocation would overflow."""
        # Return (is_safe, message)
        pass
    
    @staticmethod
    def check_stale_references(active_blocks, freed_blocks):
        """TODO: Check for use-after-free conditions."""
        # Return list of stale references
        pass
    
    @staticmethod
    def check_shape_consistency(cache_config, model_config):
        """TODO: Check that cache and model shapes match."""
        # Return (is_consistent, mismatches)
        pass

print("TODO: Implement the MemoryDiagnostics class.")
print("Each method should return clear diagnostics that would")
print("catch the corresponding bug early in the initialization phase.")

---
## Summary

In this notebook, we covered:

1. **Python debugging** with pdb/ipdb and remote debugging with debugpy
2. **GDB/CUDA-GDB** for debugging C++/CUDA extensions
3. **py-spy profiling** for identifying Python bottlenecks in production
4. **torch.profiler** for PyTorch operation profiling
5. **CUDA error debugging** for common GPU errors
6. **NCCL debugging** for distributed communication issues
7. **Common bug patterns** and diagnostic approaches

**Next:** Chapter 11.4 - Performance Profiling Workflow